# Quick demonstration

## Create the simulation

In [ ]:
import numpy as np
from lucifex.fem import grid_average, grid_cross_section, refine_grid_function
from lucifex.fdm import AB, CN, GridFunctionSeries, NumericSeries
from lucifex.io import write
from lucifex.sim import run, xdmf_to_npz
from lucifex.viz import (
    plot_colormap, create_animation, plot_line, save_figure, plot_twin_lines,
    display_animation, get_ipynb_file_name, set_ipynb_variable,
)
from lucifex.utils.fenicsx_utils import mesh_axes
from lucifex.utils.array_utils import as_index
from crocodil.dns.theory import threshold_rayleigh
from crocodil.dns.system_a import (
    dns_system_a, SYSTEM_A_REFERENCE, critical_saturation, 
    mass_dissolved_asymptote, mass_capillary_asymptote,
    mass_dissolved_initial, mass_capillary_initial,
)

STORE = 1
WRITE = None
DIR_ROOT = f'./{get_ipynb_file_name()}'
NX = set_ipynb_variable('NX', 60)
NY = set_ipynb_variable('NY', 60)
ANIM = set_ipynb_variable('ANIM', False)

simulation = dns_system_a(
    store_delta=STORE, 
    write_delta=WRITE, 
    dir_root=DIR_ROOT, 
    dir_datetime=True,
    dir_uid=True,
)(
    Nx=NX,
    Ny=NY,
    scaling='advective',
    **SYSTEM_A_REFERENCE.replace(Ra=800.0),
    D_adv=AB(1)@CN, 
    D_diff=AB(1)@CN,
    dt_max=0.1,
    courant_adv=0.75,
    courant_diff=0.75,
    courant_reac=0.1,
    c_stabilization=None,
    c_limits=True,
    diagnostic=True,
)

Lx, Ly = simulation['Lx', 'Ly']
Ra, Da, epsilon, zeta0, sr, cr = (
    float(i) for i in simulation['Ra', 'Da', 'epsilon', 'zeta0', 'sr', 'cr']
)

Ra_thresh = threshold_rayleigh(Lx, Ly, NX, 2)
print(f"Ra = {Ra} , Ra_thresh = {Ra_thresh}")

sr_crit = critical_saturation(zeta0, cr, epsilon)
print(f'sr = {sr} , sr_crit = {sr_crit}')

Ra = 800.0 , Ra_thresh = 1350.0
sr = 0.2 , sr_crit = 0.10000000000000002


## Run the simulation

In [ ]:
n_stop = set_ipynb_variable('N_STOP', 20)
t_stop = 100.0
dt_init = 1e-6
n_init = 10

write(simulation.parameters, simulation.parameter_file, simulation.dir_path, mode='w')
run(simulation, n_stop=n_stop, t_stop=t_stop, dt_init=dt_init, n_init=n_init, show_progress=True)
if WRITE: xdmf_to_npz(simulation, delete_xdmf=False)

s, c, psi, u = simulation['s', 'c', 'psi', 'u']
mC, mD, f = simulation['mC', 'mD', 'f']
fZeta0, fZetaPlus, fZetaMinus = f.split()

100%|██████████| 20/20 [00:07<00:00,  2.55it/s]


## Visualization

### Concentration

In [ ]:
_display = None
if not ANIM:
    time_indices = as_index(c.time_series, (0, 0.5, -1), fraction=True)
    for i in time_indices:
        fig, ax = plot_colormap(c.series[i], title=f'c(t={c.time_series[i]})$')
else:
    time_slice = slice(0, None, 2)
    titles = [f'${c.name}(t={t:.3f})$' for t in c.time_series[time_slice]]
    anim = create_animation(
        plot_colormap,
        colorbar=False,
    )(c.series[time_slice], title=titles)
    anim_path = save_figure(f'{c.name}(t)', simulation.dir_path, prefix_ipynb=False, return_path=True)(anim)
    _display = display_animation(anim_path)
_display

### Saturation

In [ ]:
_display = None
if not ANIM:
    time_indices = as_index(s.time_series, (0, 0.5, -1), fraction=True)
    for i in time_indices:
        fig, ax = plot_colormap(
            refine_grid_function(s.series[i], (1, 10)),
            title=f's(t={s.time_series[i]})$',
        )
else:
    time_slice = slice(0, None, 2)
    titles = [f'${s.name}(t={t:.3f})$' for t in s.time_series[time_slice]]
    anim = create_animation(
        plot_colormap,
        colorbar=(0, sr),
        y_lims=(zeta0 * Ly, Ly),
        aspect='auto',
    )(s.series[time_slice], title=titles)
    anim_path = save_figure(f'{s.name}(t)', simulation.dir_path, prefix_ipynb=False, return_path=True)(anim)
    _display = display_animation(anim_path)
_display

## Physical diagnostics

### Flux

In [ ]:
superscripts = ('', '^+', '^-')

for flx, sup in zip((fZeta0, fZetaPlus, fZetaMinus), superscripts):
    fig, ax = plot_line(
        [(flx.time_series, [np.sum(i) for i in flx.value_series]), (flx.time_series, flx.value_series)],
        cyc='black',
        x_label='$t$',
        legend_labels=['$F$', '$F_{\mathbf{u}}$', '$F_{\mathsf{D}}$'],
        legend_title=f'$y=\zeta_0{sup}$',
    )
    save_figure(f'f(y=zeta0{sup},t)', simulation.dir_path, prefix_ipynb=False)(fig)
    if np.max(fZeta0.time_series) > 50.0:
        ax.set_xscale('log')

### Mass

In [ ]:
mC_initial = mass_capillary_initial(epsilon, zeta0, sr, Lx, Ly)
mD_initial = mass_dissolved_initial(zeta0, sr, cr, Lx, Ly)
m_initial = mC_initial + mD_initial

mC_asymp = mass_capillary_asymptote(m_initial, epsilon, Lx, Ly)
mD_asymp = mass_dissolved_asymptote(m_initial, epsilon, Lx, Ly)

fig, ax = plot_line(
    (mC.time_series, mC.value_series),
    x_label='$t$',
    y_label='$m_C$',
)
ax.hlines(
    [mC_initial, mC_asymp], 
    0, max(mC.time_series), 
    linestyles='dotted', colors='black', linewidths=0.75,
)
save_figure('mC(t)', simulation.dir_path, prefix_ipynb=False)(fig)

fig, ax = plot_line(
    (mD.time_series, mD.value_series),
    x_label='$t$',
    y_label='$m_D$'
)
ax.hlines(
    [mD_initial, mD_asymp], 
    0, max(mD.time_series), 
    linestyles='dotted', colors='black', linewidths=0.75,
)
save_figure('mD(t)', simulation.dir_path, prefix_ipynb=False)(fig)

### Horizontal averages

In [ ]:
sAvgX = GridFunctionSeries(grid_average(s.series, 'x'), s.time_series, 'sAvgX')
cAvgX = GridFunctionSeries(grid_average(c.series, 'x'), c.time_series, 'cAvgX')

fig, ax = plot_line(
    sAvgX.series, 
    cyc='jet',
    legend_title='$t$',
    legend_labels=(min(sAvgX.time_series), max(sAvgX.time_series)),
    x_label='$\langle s\\rangle_x$', 
    y_label='$y$',
    flip=True,
    y_lims=(zeta0 * Ly - 0.5 * (1 - zeta0) * Ly, Ly),
)
ax.hlines(zeta0 * Ly, 0, sr, linestyles='dotted', colors='black', linewidths=0.75)
save_figure('sAvgX(t)', simulation.dir_path, prefix_ipynb=False)(fig)

fig, ax = plot_line(
    cAvgX.series, 
    cyc='jet',
    legend_title='$t$',
    legend_labels=(min(cAvgX.time_series), max(cAvgX.time_series)),
    x_label='$\langle c\\rangle_x$', 
    y_label='$y$',
    flip=True,
    y_lims=(0, Ly),
)
save_figure('cAvgX(t)', simulation.dir_path, prefix_ipynb=False)(fig)

### Subdomain averages

In [ ]:
y_axis = mesh_axes(use_cache=True)(c.mesh)[1]
zeta0_index = as_index(y_axis, zeta0)
slcPlus = slice(zeta0_index, None)
slcMinus = slice(0, zeta0_index)

cPlus = NumericSeries(
    grid_average(c.series, ('x', 'y'), (':', slcPlus)), c.time_series, 'cPlus',
)

cMinus = NumericSeries(
    grid_average(c.series, ('x', 'y'), (':', slcMinus)), c.time_series, 'cMinus',
)

sPlus = NumericSeries(
    grid_average(s.series, ('x', 'y'), (':', slcPlus)), s.time_series, 'sPlus',
)

fig, ax = plot_line(
    (cPlus.time_series, cPlus.value_series),
    x_label='$t$',
    y_label='$c^+$',
)
save_figure('cPlus(t)', simulation.dir_path, prefix_ipynb=False)(fig)

fig, ax = plot_line(
    (cMinus.time_series, cMinus.value_series),
    x_label='$t$',
    y_label='$c^-$',
)
save_figure('cMinus(t)', simulation.dir_path, prefix_ipynb=False)(fig)

fig, ax = plot_line(
    (sPlus.time_series, sPlus.value_series),
    x_label='$t$',
    y_label='$s^+$',
)
save_figure('sPlus(t)', simulation.dir_path, prefix_ipynb=False)(fig)

### Correlation

In [ ]:
time_index = 200
y_cs_target = 0.95
cx, y_cs = grid_cross_section(c.series[time_index], 'y', y_cs_target)
sx, y_cs = grid_cross_section(s.series[time_index], 'y', y_cs_target)

plot_twin_lines(
    (cx, sx),
    twin_labels=(f'$c(y={y_cs})$', f'$s(y={y_cs})$')
)

## Modelling

In [ ]:
from crocodil.modelling.system_a.flux import FluxModel, FluxModelEquations
from functools import partial

gamma = 1.0

legend_title = f'$\gamma={gamma}$\n$(\\alpha, \\beta)$'
legend_labels = ['DNS']
fDNS = fZeta0
f_lines = [(fDNS.time_series, [np.sum(i) for i in fDNS.value_series])]
mC_lines = [(mC.time_series, mC.value_series)]
mD_lines = [(mD.time_series, mD.value_series)]
cPlus_lines = [(cPlus.time_series, cPlus.value_series)]
cMinus_lines = [(cMinus.time_series, cMinus.value_series)]
sPlus_lines = [(sPlus.time_series, sPlus.value_series)]

t_ics_model = 7.0 # TODO deduce this programatically
t_ics_model_index = as_index(fDNS.time_series, t_ics_model)
t_model = fDNS.time_series[t_ics_model_index:]

ics_model = (
    cPlus.value_series[t_ics_model_index],
    cMinus.value_series[t_ics_model_index],
    sPlus.value_series[t_ics_model_index],
)

flux = lambda cp, cm, alpha, beta: (
    alpha * (cp - cm) + beta * (cp - cm) ** 2
)
alpha_opts = (0, 0.05)
beta_opts = (0.1, 0)

for alpha, beta in zip(alpha_opts, beta_opts):
    flux_model = FluxModel(
        t_model,
        Lx * Ly,
        Da,
        epsilon,
        zeta0,
        sr,
        cr,
        gamma,
        partial(flux, alpha=alpha, beta=beta),
        constraint=False,
        ics=ics_model,
    )
    f_lines.append((flux_model.t, -flux_model.f))
    mC_lines.append((flux_model.t, flux_model.mC))
    mD_lines.append((flux_model.t, flux_model.mD))
    cPlus_lines.append((flux_model.t, flux_model.cPlus))
    cMinus_lines.append((flux_model.t, flux_model.cMinus))
    sPlus_lines.append((flux_model.t, flux_model.sPlus))
    legend_labels.append(f'({alpha}, {beta})')


add_start_line = lambda ax:(
    ax.vlines(t_ics_model, *ax.get_ylim(), linestyles='dotted', colors='red')
)

fig, ax = plot_line(
    f_lines,
    y_label='$F$',
    legend_labels=legend_labels,  legend_title=legend_title,
)
add_start_line(ax)

fig, ax = plot_line(
    mC_lines,
    y_label='$m_C$',
    legend_labels=legend_labels, legend_title=legend_title,
)
add_start_line(ax)

fig, ax = plot_line(
    mD_lines,
    y_label='$m_D$',
    legend_labels=legend_labels, legend_title=legend_title,
)
add_start_line(ax)

fig, ax = plot_line(
    cPlus_lines,
    y_label='$c^+$',
    legend_labels=legend_labels, legend_title=legend_title, #y_lims=(0.9, 1.0),
)
add_start_line(ax)

fig, ax = plot_line(
    cMinus_lines,
    y_label='$c^-$',
    legend_labels=legend_labels, legend_title=legend_title,
)
add_start_line(ax)

fig, ax = plot_line(
    sPlus_lines,
    y_label='$s^+$',
    legend_labels=legend_labels, legend_title=legend_title,
)
add_start_line(ax)

## Numerical diagnostics

### Timestep contraints

In [ ]:
dt, dtU, dtD, dtSigma = simulation['dt', 'dtU', 'dtD', 'dtSigma']

fig, ax = plot_line(
    [
        (dt.time_series, dt.value_series), 
        (dtU.time_series, dtU.value_series), 
        (dtD.time_series, dtD.value_series), 
        (dtSigma.time_series, dtSigma.value_series),
    ],
    x_label='$t$',
    legend_labels=['$\Delta t$', '$\Delta t_{\mathbf{u}}$', '$\Delta t_{\mathsf{D}}$', '$\Delta t_{\Sigma}$'],
)
ax.set_yscale('log')
save_figure('dt(t)', simulation.dir_path, prefix_ipynb=False)(fig)

### Velocity norms

In [ ]:
uRMS, uMinMax, uDiv = simulation['uRMS', 'uMinMax', 'uDiv']
uMax = uMinMax.sub(1)

fig, ax = plot_line(
    (uRMS.time_series, uRMS.value_series),
    x_label='$t$',
    y_label='$\mathrm{rms}(\mathbf{u})$',
)
ax.set_yscale('log')
save_figure('uRMS(t)', simulation.dir_path, prefix_ipynb=False)(fig)

fig, ax = plot_line(
    (uMax.time_series, uMax.value_series),
    x_label='$t$',
    y_label='$\max_{\mathbf{x}}|\mathbf{u}|$',
)
ax.set_yscale('log')
save_figure('uMax(t)', simulation.dir_path, prefix_ipynb=False)(fig)

fig, ax = plot_line(
    (uDiv.time_series, uDiv.value_series),
    x_label='$t$',
    y_label='$\mathrm{divnorm}(\mathbf{u})$',
)
save_figure('uDiv(t)', simulation.dir_path, prefix_ipynb=False)(fig)

### Concentration limits

In [ ]:
cMinMax = simulation['cMinMax']
cMin, cMax = cMinMax.split()

fig, ax = plot_line(
    (cMax.time_series, cMax.value_series),
    x_label='$t$',
    y_label='$\max_{\mathbf{x}}c$',
    y_lims=(1-0.1, 1+0.1),
)
save_figure('cMax(t)', simulation.dir_path, prefix_ipynb=False)(fig)

fig, ax = plot_line(
    (cMin.time_series, cMin.value_series),
    x_label='$t$',
    y_label='$\min_{\mathbf{x}}c$',
)
save_figure('cMin(t)', simulation.dir_path, prefix_ipynb=False)(fig)

if 'cCorr' in simulation.namespace:
    cCorr = simulation['cCorr']
    fig, ax = plot_line(
        [
            (cCorr.time_series, [np.min(i) for i in cCorr.dofs_series]),
            (cCorr.time_series, [np.max(i) for i in cCorr.dofs_series]),
        ],
        x_label='$t$',
        legend_labels=['$\min_{\mathbf{x}}\mathcal{C}(c)$', '$\max_{\mathbf{x}}\mathcal{C}(c)$'],
    )
    save_figure('cCorrMinMax(t)', simulation.dir_path, prefix_ipynb=False)(fig)

### Mass conservation

In [ ]:
m_sum = np.array([i + j for i, j in zip(mC.value_series, mD.value_series)]) - 1

fig, ax = plot_line(
    (mC.time_series, m_sum / m_sum[0]),
    x_label='$t$',
    y_label='$(m_C + m_D)/m_0 - 1$'
)
save_figure('m(t)_normalized', simulation.dir_path, prefix_ipynb=False)(fig)